# SHM — Cumulative Fatigue Damage Prediction

This notebook provides a complete workflow for the Structural Health Monitoring (SHM) task:

1. Import and prepare the training data
2. Extract stress-signal and cycle-related features
3. Train and evaluate three regression models
4. Compare the models using 5-fold cross-validation MAPE
5. Select the best model automatically
6. Retrain the best model on all training data
7. Apply it to the unseen test set
8. Export `shm_predictions.csv`
9. Produce simple plots for the final dashboard

**Target:** one numeric cumulative-fatigue-damage prediction per test CSV.

**Primary evaluation metric:** MAPE. Lower MAPE is better.

## 1. Imports and file paths

Before running the notebook, make sure your Colab folders look like:

```text
/content/
├── Train_Labels.csv
├── SHM_Train/
│   ├── train01.csv
│   ├── ...
│   └── train64.csv
└── SHM_Test/
    ├── test01.csv
    ├── ...
    └── test16.csv
```

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import find_peaks

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor
)

TRAIN_FOLDER = "/content/SHM_Train"
TEST_FOLDER = "/content/SHM_Test"
LABELS_PATH = "/content/Train_Labels.csv"
OUTPUT_PATH = "/content/shm_predictions.csv"

RANDOM_STATE = 42
N_SPLITS = 5

## 2. Load one stress signal

Each SHM CSV contains one long stress signal and has no header row.

In [ ]:
def load_signal(filepath):
    df = pd.read_csv(filepath, header=None)

    signal = pd.to_numeric(
        df.iloc[:, 0],
        errors="coerce"
    ).dropna().to_numpy(dtype=float)

    if len(signal) == 0:
        raise ValueError(f"No valid numeric stress values found in {filepath}")

    return signal

## 3. Feature engineering

Each raw CSV contains hundreds of thousands of stress readings.  
The model needs one row per file, so the signal is summarised into:

- basic statistics: mean, standard deviation, min, max, range, RMS
- distribution statistics: percentiles and IQR
- change statistics: how rapidly stress changes
- peak/trough statistics
- approximate cycle-range statistics

The cycle features are not a full rainflow implementation; they are practical signal features designed to capture repeated loading behaviour.

In [ ]:
def extract_cycle_features(signal):
    # Local turning points
    peaks, _ = find_peaks(signal)
    troughs, _ = find_peaks(-signal)

    turning_points = np.sort(
        np.concatenate([peaks, troughs])
    )

    if len(turning_points) < 2:
        return {
            "cycle_count": 0,
            "mean_cycle_range": 0.0,
            "median_cycle_range": 0.0,
            "max_cycle_range": 0.0,
            "std_cycle_range": 0.0,
            "q75_cycle_range": 0.0,
            "q90_cycle_range": 0.0,
            "q95_cycle_range": 0.0,
            "q99_cycle_range": 0.0,
            "large_cycle_count": 0
        }

    turning_values = signal[turning_points]
    cycle_ranges = np.abs(np.diff(turning_values))

    q75, q90, q95, q99 = np.percentile(
        cycle_ranges,
        [75, 90, 95, 99]
    )

    return {
        "cycle_count": len(cycle_ranges),
        "mean_cycle_range": np.mean(cycle_ranges),
        "median_cycle_range": np.median(cycle_ranges),
        "max_cycle_range": np.max(cycle_ranges),
        "std_cycle_range": np.std(cycle_ranges),
        "q75_cycle_range": q75,
        "q90_cycle_range": q90,
        "q95_cycle_range": q95,
        "q99_cycle_range": q99,
        "large_cycle_count": int(np.sum(cycle_ranges >= q90))
    }

In [ ]:
def extract_features(filepath):
    signal = load_signal(filepath)
    diff_signal = np.diff(signal)

    q01, q05, q25, q50, q75, q95, q99 = np.percentile(
        signal,
        [1, 5, 25, 50, 75, 95, 99]
    )

    prominence = np.std(signal)

    peaks, _ = find_peaks(
        signal,
        prominence=prominence
    )

    troughs, _ = find_peaks(
        -signal,
        prominence=prominence
    )

    features = {
        "filename": os.path.basename(filepath),

        # Signal size / level
        "n_samples": len(signal),
        "mean": np.mean(signal),
        "std": np.std(signal),
        "min": np.min(signal),
        "max": np.max(signal),
        "range": np.ptp(signal),
        "rms": np.sqrt(np.mean(signal ** 2)),
        "median": q50,

        # Distribution
        "q01": q01,
        "q05": q05,
        "q25": q25,
        "q75": q75,
        "q95": q95,
        "q99": q99,
        "iqr": q75 - q25,

        # Magnitude / movement
        "mean_absolute_stress": np.mean(np.abs(signal)),
        "max_absolute_stress": np.max(np.abs(signal)),
        "mean_absolute_change": np.mean(np.abs(diff_signal)) if len(diff_signal) else 0.0,
        "std_change": np.std(diff_signal) if len(diff_signal) else 0.0,
        "max_absolute_change": np.max(np.abs(diff_signal)) if len(diff_signal) else 0.0,

        # Crossings and prominent turning points
        "zero_crossings": int(np.sum(np.diff(np.signbit(signal)) != 0)),
        "peak_count": len(peaks),
        "trough_count": len(troughs),
        "peak_rate": len(peaks) / len(signal),
        "trough_rate": len(troughs) / len(signal)
    }

    if len(peaks) > 0:
        peak_values = signal[peaks]
        features.update({
            "mean_peak_value": np.mean(peak_values),
            "max_peak_value": np.max(peak_values),
            "std_peak_value": np.std(peak_values)
        })
    else:
        features.update({
            "mean_peak_value": 0.0,
            "max_peak_value": 0.0,
            "std_peak_value": 0.0
        })

    if len(troughs) > 0:
        trough_values = signal[troughs]
        features.update({
            "mean_trough_value": np.mean(trough_values),
            "min_trough_value": np.min(trough_values),
            "std_trough_value": np.std(trough_values)
        })
    else:
        features.update({
            "mean_trough_value": 0.0,
            "min_trough_value": 0.0,
            "std_trough_value": 0.0
        })

    features.update(extract_cycle_features(signal))

    return features

## 4. Build the training feature table

This converts every `trainXX.csv` into one row of features and merges the row with its known `damage` label.

In [ ]:
train_files = sorted([
    f for f in os.listdir(TRAIN_FOLDER)
    if f.lower().endswith(".csv")
])

print("Training files found:", len(train_files))
print(train_files[:10])

In [ ]:
feature_rows = []

for i, filename in enumerate(train_files, start=1):
    filepath = os.path.join(TRAIN_FOLDER, filename)

    print(f"Processing {i}/{len(train_files)}: {filename}")

    feature_rows.append(
        extract_features(filepath)
    )

features_df = pd.DataFrame(feature_rows)

labels_df = pd.read_csv(LABELS_PATH)

dataset = features_df.merge(
    labels_df,
    on="filename",
    how="left"
)

if dataset["damage"].isna().any():
    missing = dataset.loc[dataset["damage"].isna(), "filename"].tolist()
    raise ValueError(f"Missing damage labels for: {missing}")

print("\nTraining dataset shape:", dataset.shape)
display(dataset.head())

In [ ]:
# Optional: save the engineered training table for inspection
TRAIN_FEATURES_PATH = "/content/SHM_features.csv"
dataset.to_csv(TRAIN_FEATURES_PATH, index=False)

print("Saved engineered training data to:", TRAIN_FEATURES_PATH)

## 5. Prepare X and y

- `X` = engineered stress features
- `y` = true cumulative fatigue damage

The filename is only an identifier and is not used as a model feature.

In [ ]:
X = dataset.drop(columns=["filename", "damage"])
y = dataset["damage"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of model features:", X.shape[1])

## 6. Define the three regression models

All three models are evaluated using the **same 5-fold cross-validation splits** so the comparison is fair.

In [ ]:
models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_STATE
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE
    )
}

models

## 7. Evaluate all three models using 5-fold MAPE

For each fold:

1. train on 4/5 of the files
2. predict the remaining 1/5
3. calculate MAPE
4. repeat 5 times
5. average the five MAPEs

Lower MAPE is better.

In [ ]:
kf = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

comparison_rows = []
fold_results = {}

for model_name, model in models.items():

    fold_mapes = []

    print(f"\n{model_name}")
    print("-" * len(model_name))

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]

        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        model.fit(X_train, y_train)

        y_pred = model.predict(X_val)

        fold_mape = mean_absolute_percentage_error(
            y_val,
            y_pred
        )

        fold_mapes.append(fold_mape)

        print(
            f"Fold {fold}: {fold_mape * 100:.2f}% MAPE"
        )

    average_mape = np.mean(fold_mapes)
    competition_score = max(0, 1 - average_mape)

    fold_results[model_name] = fold_mapes

    comparison_rows.append({
        "Model": model_name,
        "Average MAPE": average_mape,
        "Average MAPE %": average_mape * 100,
        "Approx. Competition Score": competition_score
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    "Average MAPE",
    ascending=True
).reset_index(drop=True)

display(comparison_df)

## 8. Compare the models and select the best one automatically

The best model is the one with the **lowest average 5-fold MAPE**.

In [ ]:
best_model_name = comparison_df.loc[0, "Model"]
best_mape = comparison_df.loc[0, "Average MAPE"]
best_score = comparison_df.loc[0, "Approx. Competition Score"]

print("Best model:", best_model_name)
print(f"Average MAPE: {best_mape * 100:.2f}%")
print(f"Approx. competition score: {best_score:.4f}")

In [ ]:
# Simple model-comparison graph
plt.figure(figsize=(8, 5))

plt.bar(
    comparison_df["Model"],
    comparison_df["Average MAPE %"]
)

plt.title("Model Comparison — 5-Fold Cross-Validation")
plt.ylabel("Average MAPE (%)")
plt.xlabel("Model")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 9. Inspect the best model's validation predictions

This creates out-of-fold predictions, meaning each training file is predicted by a model that did not train on that file.

This is useful for showing:

- actual damage
- predicted damage
- percentage error for each file

In [ ]:
best_model_template = models[best_model_name]

oof_predictions = np.zeros(len(X), dtype=float)

for train_idx, val_idx in kf.split(X):
    # Create a fresh copy of the selected model
    if best_model_name == "Random Forest":
        fold_model = RandomForestRegressor(
            n_estimators=300,
            random_state=RANDOM_STATE
        )
    elif best_model_name == "Gradient Boosting":
        fold_model = GradientBoostingRegressor(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            random_state=RANDOM_STATE
        )
    else:
        fold_model = ExtraTreesRegressor(
            n_estimators=300,
            random_state=RANDOM_STATE
        )

    fold_model.fit(
        X.iloc[train_idx],
        y.iloc[train_idx]
    )

    oof_predictions[val_idx] = fold_model.predict(
        X.iloc[val_idx]
    )

validation_results = pd.DataFrame({
    "filename": dataset["filename"],
    "Actual": y.values,
    "Predicted": oof_predictions
})

validation_results["Error %"] = (
    np.abs(
        validation_results["Actual"]
        - validation_results["Predicted"]
    )
    / np.abs(validation_results["Actual"])
) * 100

validation_results = validation_results.sort_values(
    "Error %",
    ascending=False
).reset_index(drop=True)

display(validation_results)

In [ ]:
overall_oof_mape = mean_absolute_percentage_error(
    validation_results["Actual"],
    validation_results["Predicted"]
)

print(f"Overall out-of-fold MAPE: {overall_oof_mape * 100:.2f}%")
print(f"Competition-style score: {max(0, 1 - overall_oof_mape):.4f}")

## 10. Fit the best model on all training data

Cross-validation was used only to choose the model.  
For the final submission, the selected model is retrained using **all labelled training files**.

In [ ]:
if best_model_name == "Random Forest":
    final_model = RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE
    )

elif best_model_name == "Gradient Boosting":
    final_model = GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_STATE
    )

else:
    final_model = ExtraTreesRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE
    )

final_model.fit(X, y)

print(f"Final {best_model_name} model trained on all {len(X)} training files.")

## 11. Optional: feature importance of the final model

All three selected model types support `feature_importances_`.

This is useful for the technical write-up, but it does not need to be the main visual shown to a non-technical dashboard user.

In [ ]:
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": final_model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

display(importance_df.head(15))

## 12. Process the unseen test data

The test files must use the **exact same feature extraction pipeline** as the training files.

In [ ]:
test_files = sorted([
    f for f in os.listdir(TEST_FOLDER)
    if f.lower().endswith(".csv")
])

print("Test files found:", len(test_files))
print(test_files)

In [ ]:
test_feature_rows = []

for i, filename in enumerate(test_files, start=1):
    filepath = os.path.join(TEST_FOLDER, filename)

    print(f"Processing {i}/{len(test_files)}: {filename}")

    test_feature_rows.append(
        extract_features(filepath)
    )

test_features_df = pd.DataFrame(test_feature_rows)

display(test_features_df.head())

## 13. Align test features and predict cumulative fatigue damage

The feature columns are explicitly aligned to the training columns before prediction.  
This prevents accidental column-order differences between training and test data.

In [ ]:
X_test = test_features_df.drop(
    columns=["filename"]
)

# Force identical feature order to training
X_test = X_test[X.columns]

print("Training feature count:", X.shape[1])
print("Test feature count:", X_test.shape[1])

if list(X_test.columns) != list(X.columns):
    raise ValueError("Training and test feature columns do not match.")

test_predictions = final_model.predict(X_test)

print("Predictions generated:", len(test_predictions))

## 14. Final SHM results

The required output contains:

- `file_id` — original test filename
- `prediction` — predicted cumulative fatigue damage

In [ ]:
submission = pd.DataFrame({
    "file_id": test_features_df["filename"],
    "prediction": test_predictions
})

display(submission)

In [ ]:
submission.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Final submission saved to:")
print(OUTPUT_PATH)

## 15. Dashboard visual — dynamic stress signal

This plot can be used in the final app/dashboard to show the stress pattern for an uploaded file.

The signal is downsampled **only for display**.  
The model still extracts features from the full original signal.

In [ ]:
def plot_stress_signal(filepath, max_points=5000):
    signal = load_signal(filepath)

    if len(signal) > max_points:
        indices = np.linspace(
            0,
            len(signal) - 1,
            max_points
        ).astype(int)

        display_x = indices
        display_signal = signal[indices]

    else:
        display_x = np.arange(len(signal))
        display_signal = signal

    plt.figure(figsize=(14, 5))

    plt.plot(
        display_x,
        display_signal,
        linewidth=0.8
    )

    plt.title(
        f"Dynamic Stress Signal — {os.path.basename(filepath)}"
    )

    plt.xlabel("Measurement Sequence")
    plt.ylabel("Stress")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Example dashboard plot
# Change the filename if needed.
example_test_file = os.path.join(
    TEST_FOLDER,
    test_files[0]
)

plot_stress_signal(example_test_file)

## Final workflow summary

```text
64 labelled training stress files
            ↓
Feature engineering
            ↓
Random Forest
Gradient Boosting
Extra Trees
            ↓
Same 5-fold MAPE comparison
            ↓
Automatically select lowest-MAPE model
            ↓
Retrain selected model on all training files
            ↓
Extract identical features from 16 test files
            ↓
Predict cumulative fatigue damage
            ↓
shm_predictions.csv
```

The final submission file for the SHM component is `shm_predictions.csv`.

MAPE is used to evaluate the model; it is **not** the final prediction output.